In [99]:
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from atlassian import Jira

# model="llama3.1:8b-instruct-q8_0"
model="llama3.1"
llm = ChatOllama(model=model, temperature=0)

In [51]:
from dotenv import load_dotenv
import os
load_dotenv()
try:
    jira = Jira(
    url=os.getenv("url"),
    username=os.getenv("username"),
    password=os.getenv("password"),
    cloud=True)
except Exception as e:
    print(f"Unable to login to jira, Error: {e}")

In [52]:
import pandas as pd

def createJiraTaskFromLocalCSVFile(csvPath:str):
    df = pd.read_csv(csvPath)
    for index, row in df.iterrows():
        if pd.notna(row['summary']) & pd.isna(row['jira']):
            fields = {'project':{'key':'AN30'},'issuetype': {'name': 'Task'},'summary': row['summary'], 'description':row['description'], 'assignee':{'id':row['assignee']}}

            # res will contain {'id': '2784859', 'key': 'AN30-6067', 'self': 'link to the json response'}
            res=jira.issue_create(fields)

            df.at[index, 'jira'] = f'https://amagiengg.atlassian.net/browse/{res['key']}'
    df.to_csv(csvPath, index=False)
    
# createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv') 
# key=2784795

def getIssue():
    print(jira.issue(key))

# getIssue()


In [152]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

store = {}

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a highly capable AI assistant tasked with understanding user queries and responding with the appropriate function from the list provided below. Your response must adhere to the following constraints:

            Functions:
                - createJiraTaskFromLocalCSVFile(pathToCSV)

            Constraints:
                - Only use the functions listed above. Do not generate or suggest any other functions.
                - Ensure that the function you generate directly addresses the query made by the user.
                - If the query does not correspond to any function you are allowed to use, respond with an empty string ''.
                - Include only the function name and arguments in your response, without any additional text.
                - If there is no path mentioned in the query then respond with empty string ''.
                - If the question is related to any task we did in the current session then you should give relevant answer.

            Examples:
                - Query: "Can you read the csv in the path '../csvFiles/jira_task_list.csv' and create jira for each items" 
                Response: "createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')"
                - Query: "read the csv in the path '../csvFiles/jira_task_list.csv' and create jira for each items" 
                Response: "createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')"
                - Query: "csv read jira '../csvFiles/jira_task_list.csv'"
                Response: ''
                - Query: "csv read '../csvFiles/jira_task_list.csv'"
                Response: ''
                - Query: "jira tickets '../csvFiles/jira_task_list.csv'"
                Response: ''

            Be mindful that only the functions defined above are valid, and the response must match the function signature exactly.
            """
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain =  RunnablePassthrough.assign(messages=itemgetter("messages")) | prompt | llm 

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

config = {"configurable": {"session_id": "new"}}


In [153]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'")]},
    config=config,
)
res = response.content.strip()
print(res)



# for r in with_message_history.invoke(
#     {"messages":[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'")]},
#     config=config,
# ):
#     print(r.content, end="", flush=True)

createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')


In [154]:
print(store)

{'new': InMemoryChatMessageHistory(messages=[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'"), AIMessage(content="createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')", response_metadata={'model': 'llama3.1', 'created_at': '2024-08-24T17:32:49.808062Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3747059125, 'load_duration': 28551083, 'prompt_eval_count': 398, 'prompt_eval_duration': 3181510000, 'eval_count': 18, 'eval_duration': 532159000}, id='run-fdafdab9-ca4a-4cb5-b13d-e731ae1e62bf-0')])}


In [155]:
from langchain_core.messages import SystemMessage

def updateSessionHistory(session_id: str):
    if session_id in store:
        get_session_history(session_id).add_message(SystemMessage("""We have created the following jira \n- Have popover with the same width as the field width: \n- Build a dynamic deployment action for particular feature branch from PR itself: """))

updateSessionHistory('new')

In [156]:
print(store)

{'new': InMemoryChatMessageHistory(messages=[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'"), AIMessage(content="createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')", response_metadata={'model': 'llama3.1', 'created_at': '2024-08-24T17:32:49.808062Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3747059125, 'load_duration': 28551083, 'prompt_eval_count': 398, 'prompt_eval_duration': 3181510000, 'eval_count': 18, 'eval_duration': 532159000}, id='run-fdafdab9-ca4a-4cb5-b13d-e731ae1e62bf-0'), SystemMessage(content='We have created the following jira \n- Have popover with the same width as the field width: https://amagiengg.atlassian.net/browse/AN30-6067\n- Build a dynamic deployment action for particular feature branch from PR itself: https://amagiengg.atlassian.net/browse/AN30-6068')])}


In [ ]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="which all jira have we created")]},
    config=config,
)
res = response.content.strip()
print(res)

In [163]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="how many jira did we create")]},
    config=config,
)
res = response.content.strip()
print(res)

3


In [ ]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="can you please share those?")]},
    config=config,
)
res = response.content.strip()
print(res)

In [160]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list1.csv'")]},
    config=config,
)
res = response.content.strip()
print(res)


createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list1.csv')


In [161]:
def updateSessionHistory(session_id: str):
    if session_id in store:
        get_session_history(session_id).add_message(SystemMessage("""We have created the following jira \n- Have popover with the same width as the field width: \n- Build a dynamic deployment action for particular feature branch from PR itself: \n- Show delivery status inside delivery details: """))

updateSessionHistory('new')

In [132]:

if(len(res)>2):
    try:
        eval(res)
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Unexpected response:", res)

Unexpected response: ''


In [29]:
ar=[1,2,3]
print(len(ar))

3
